There is no simple list of patches anywhere to be used, so I need to make my own one. This notebook opens https://www.leagueoflegends.com/en-us/news/tags/patch-notes/, clicks **SHOW MORE** until the full archive is loaded, then extracts **patch_number**, **patch_date**, and **patch_url** from the final HTML.

In [2]:
import re
import time

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [3]:
url = "https://www.leagueoflegends.com/en-us/news/tags/patch-notes/"

chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1920,1080")


def parse_patch_number(title_text: str) -> str:
    match = re.search(r"Patch\s+([0-9]+(?:\.[0-9]+)?)", title_text, re.IGNORECASE)
    return match.group(1) if match else title_text.strip()


driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 10)
html = ""
patch_rows = []

try:
    driver.get(url)

    while True:
        try:
            show_more_button = wait.until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'cta')][.//span[normalize-space()='SHOW MORE'] or normalize-space()='SHOW MORE']"))
            )
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", show_more_button)
            time.sleep(0.5)
            previous_count = len(driver.find_elements(By.CSS_SELECTOR, "[data-testid='card-title']"))
            show_more_button.click()
            wait.until(lambda current_driver: len(current_driver.find_elements(By.CSS_SELECTOR, "[data-testid='card-title']")) > previous_count)
        except (TimeoutException, NoSuchElementException):
            break

    html = driver.page_source

    for title_element in driver.find_elements(By.CSS_SELECTOR, "[data-testid='card-title']"):
        card_link = title_element.find_element(By.XPATH, "./ancestor::a[@href][1]")
        date_elements = card_link.find_elements(By.CSS_SELECTOR, "[data-testid='card-date']")
        title_text = title_element.text.strip()
        patch_rows.append(
            {
                "patch_number": parse_patch_number(title_text),
                "patch_date": date_elements[0].text.strip() if date_elements else None,
                "patch_url": card_link.get_attribute("href"),
                "patch_title": title_text,
            }
        )
finally:
    driver.quit()

patches_df = (
    pd.DataFrame(patch_rows)
    .drop_duplicates(subset=["patch_url"])
    .reset_index(drop=True)
)

patches_df

,patch_number,patch_date,patch_url,patch_title
0,26.13,6/23/2026,https://www.leagueoflegends.com/en-us/news/gam...,League of Legends Patch 26.13 Notes
1,26.12,6/9/2026,https://www.leagueoflegends.com/en-us/news/gam...,League of Legends Patch 26.12 Notes
2,26.11,5/27/2026,https://www.leagueoflegends.com/en-us/news/gam...,League of Legends Patch 26.11 Notes
3,26.10,5/12/2026,https://www.leagueoflegends.com/en-us/news/gam...,League of Legends Patch 26.10 Notes
4,26.9,4/28/2026,https://www.leagueoflegends.com/en-us/news/gam...,League of Legends Patch 26.9 Notes
...,...,...,...,...
163,9.19,9/23/2019,https://www.leagueoflegends.com/en-us/news/gam...,Patch 9.19 notes
164,9.18,9/4/2019,https://www.leagueoflegends.com/en-us/news/gam...,Patch 9.18 Notes
165,9.3,2/5/2019,https://www.leagueoflegends.com/en-us/news/gam...,Patch 9.3 notes
166,9.2,1/23/2019,https://www.leagueoflegends.com/en-us/news/gam...,Patch 9.2 notes


With the info scraped, save the dataset and then try to download the *patch highlights* image

In [ ]:
patches_df.to_csv("bronze/datasets/patches/scraped_patch_notes.csv", index=False)

In [ ]:
from pathlib import Path
import re
from html import unescape
from urllib.parse import urlparse

import requests


output_dir = Path("bronze/datasets/patches/patch_highlights_images")
output_dir.mkdir(parents=True, exist_ok=True)

image_url_pattern = re.compile(r"https://cmsassets\.rgpub\.io/sanity/images/[^\s\"'<>\)]+", re.IGNORECASE)
skins_anchor_pattern = re.compile(
    r"<a[^>]*class=['\"][^'\"]*skins[^'\"]*cboxElement[^'\"]*['\"][^>]*href=['\"]([^'\"]+)['\"]",
    re.IGNORECASE,
)


def sanitize_patch_number(value: str) -> str:
    return re.sub(r"[^0-9A-Za-z._-]+", "_", str(value).strip())


def normalize_url(url: str) -> str:
    cleaned = url.strip()
    if cleaned.startswith("//"):
        return f"https:{cleaned}"
    return cleaned


def guess_extension(image_url: str | None, content_type: str | None) -> str:
    if content_type:
        content_type = content_type.lower()
        if "png" in content_type:
            return ".png"
        if "jpeg" in content_type or "jpg" in content_type:
            return ".jpg"

    if image_url:
        path = urlparse(image_url).path.lower()
        if path.endswith(".png"):
            return ".png"
        if path.endswith(".jpg") or path.endswith(".jpeg"):
            return ".jpg"

    return ".jpg"


def extract_highlight_urls_from_skins_anchor(page_html: str) -> list[str]:
    normalized = unescape(page_html).replace("\\/", "/")
    matches = skins_anchor_pattern.findall(normalized)

    unique_urls = []
    seen = set()
    for url in matches:
        clean = normalize_url(url)
        if clean not in seen:
            seen.add(clean)
            unique_urls.append(clean)

    return unique_urls


def extract_cms_candidate_image_urls(page_html: str) -> list[str]:
    normalized = unescape(page_html).replace("\\/", "/")
    found = image_url_pattern.findall(normalized)

    unique_urls = []
    seen = set()
    for url in found:
        clean = normalize_url(url.split("\"")[0].split("'")[0].rstrip(",)"))
        if clean not in seen:
            seen.add(clean)
            unique_urls.append(clean)

    return unique_urls


def download_first_valid_image(patch_url: str, timeout: int = 20) -> tuple[bytes | None, str | None, str | None]:
    page_response = requests.get(patch_url, timeout=timeout)
    page_response.raise_for_status()

    # First try explicit highlight links from anchor tags.
    candidates = extract_highlight_urls_from_skins_anchor(page_response.text)

    # Fallback: scan page for CMS image URLs.
    if not candidates:
        candidates = extract_cms_candidate_image_urls(page_response.text)

    for candidate_url in candidates:
        try:
            image_response = requests.get(candidate_url, timeout=timeout)
            image_response.raise_for_status()
            content_type = image_response.headers.get("Content-Type", "")
            if content_type.startswith("image/"):
                return image_response.content, candidate_url, content_type
        except requests.RequestException:
            continue

    return None, None, None


for patch in patches_df.itertuples(index=False):
    patch_number = sanitize_patch_number(getattr(patch, "patch_number", "unknown"))
    patch_url = getattr(patch, "patch_url", None)

    if not patch_url:
        print(f"Skipping {patch_number}: missing patch URL")
        continue

    try:
        image_bytes, image_url, content_type = download_first_valid_image(patch_url)

        if image_bytes is None:
            print(f"No highlight image found for patch {patch_number}")
            continue

        extension = guess_extension(image_url, content_type)
        output_path = output_dir / f"{patch_number}_patch_higlights{extension}"
        output_path.write_bytes(image_bytes)

        print(f"Downloaded {patch_number} -> {output_path}")
    except requests.RequestException as exc:
        print(f"Failed {patch_number}: {exc}")

Downloaded 26.13 -> datasets\patch_highlights\26.13_patch_higlights.jpg
Downloaded 26.12 -> datasets\patch_highlights\26.12_patch_higlights.png
Downloaded 26.11 -> datasets\patch_highlights\26.11_patch_higlights.png
Downloaded 26.10 -> datasets\patch_highlights\26.10_patch_higlights.png
Downloaded 26.9 -> datasets\patch_highlights\26.9_patch_higlights.jpg
Downloaded 26.8 -> datasets\patch_highlights\26.8_patch_higlights.png
Downloaded 26.7 -> datasets\patch_highlights\26.7_patch_higlights.png
Downloaded 26.6 -> datasets\patch_highlights\26.6_patch_higlights.jpg
Downloaded 26.5 -> datasets\patch_highlights\26.5_patch_higlights.png
Downloaded 26.4 -> datasets\patch_highlights\26.4_patch_higlights.png
Downloaded 26.3 -> datasets\patch_highlights\26.3_patch_higlights.jpg
Downloaded 26.2 -> datasets\patch_highlights\26.2_patch_higlights.png
Downloaded 26.1 -> datasets\patch_highlights\26.1_patch_higlights.png
Downloaded 25.24 -> datasets\patch_highlights\25.24_patch_higlights.jpg
Downloaded

Having all the patch highlights, then I only need to make Gemini works analizing every one and saving the info in a dataset

In [ ]:
from pathlib import Path
import base64
import json
import os
from dotenv import load_dotenv
import re

import pandas as pd
from google import genai

load_dotenv()

# Prefer environment variable to avoid hardcoding secrets in notebooks.
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("Set GEMINI_API_KEY in your environment before running this cell.")

client = genai.Client(api_key=gemini_api_key)

images_dir = Path("bronze/datasets/patches/patch_highlights")
output_csv = Path("bronze/datasets/patches/patch_highlights_champion_changes.csv")

image_paths = sorted(
    [
        p
        for p in images_dir.glob("*")
        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
    ]
)

if not image_paths:
    raise FileNotFoundError(f"No images found in {images_dir}")


def infer_patch_number_from_filename(path: Path) -> str:
    match = re.match(r"(.+?)_patch_higlights", path.stem, flags=re.IGNORECASE)
    return match.group(1) if match else path.stem


def mime_from_suffix(path: Path) -> str:
    ext = path.suffix.lower()
    if ext == ".png":
        return "image/png"
    if ext in {".jpg", ".jpeg"}:
        return "image/jpeg"
    if ext == ".webp":
        return "image/webp"
    return "application/octet-stream"


records = []
errors = []

for image_path in image_paths:
    patch_number = infer_patch_number_from_filename(image_path)

    try:
        image_bytes = image_path.read_bytes()
        interaction = client.interactions.create(
            model="gemini-3.5-flash",
            input=[
                {
                    "type": "text",
                    "text": (
                        "Analyze this League of Legends patch highlight image. "
                        "Return ONLY valid JSON in this exact format: "
                        "{\"changes\": [{\"champion\": \"...\", \"change_type\": \"Buff|Nerf\", \"patch_number\": \"...\"}]} "
                        f"Use patch_number = '{patch_number}'. "
                        "If no champion buffs/nerfs are visible, return {\"changes\": []}."
                    ),
                },
                {
                    "type": "image",
                    "data": base64.b64encode(image_bytes).decode("utf-8"),
                    "mime_type": mime_from_suffix(image_path),
                },
            ],
        )

        raw_text = interaction.output_text.strip()

        # Handle occasional fenced JSON responses.
        if raw_text.startswith("```"):
            raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text, flags=re.IGNORECASE)
            raw_text = re.sub(r"\s*```$", "", raw_text)

        payload = json.loads(raw_text)
        changes = payload.get("changes", [])

        for change in changes:
            champion = str(change.get("champion", "")).strip()
            change_type = str(change.get("change_type", "")).strip().title()
            patch_num = str(change.get("patch_number", patch_number)).strip() or patch_number

            if champion and change_type in {"Buff", "Nerf"}:
                records.append(
                    {
                        "Champion": champion,
                        "Buff/Nerf": change_type,
                        "Patch Number": patch_num,
                        "Image File": image_path.name,
                    }
                )

        print(f"Processed {image_path.name}: {len(changes)} raw changes")

    except Exception as exc:
        errors.append({"image_file": image_path.name, "error": str(exc)})
        print(f"Failed {image_path.name}: {exc}")


results_df = pd.DataFrame(records).drop_duplicates().reset_index(drop=True)
results_df.to_csv(output_csv, index=False)

print(f"Saved {len(results_df)} rows to {output_csv}")

if errors:
    errors_df = pd.DataFrame(errors)
    errors_csv = Path("bronze/datasets/patches/patch_highlights_processing_errors.csv")
    errors_df.to_csv(errors_csv, index=False)
    print(f"Saved {len(errors_df)} errors to {errors_csv}")

results_df.head()

Processed 10.10_patch_higlights.png: 14 raw changes
Processed 10.11_patch_higlights.jpg: 9 raw changes
Processed 10.12_patch_higlights.jpg: 11 raw changes
Processed 10.13_patch_higlights.png: 15 raw changes
Processed 10.14_patch_higlights.jpg: 12 raw changes
Processed 10.15_patch_higlights.jpg: 14 raw changes
Processed 10.16_patch_higlights.jpg: 27 raw changes
Processed 10.17_patch_higlights.png: 12 raw changes
Processed 10.18_patch_higlights.jpg: 12 raw changes
Processed 10.19_patch_higlights.png: 14 raw changes
Processed 10.20_patch_higlights.jpg: 14 raw changes
Processed 10.21_patch_higlights.png: 12 raw changes
Processed 10.22_patch_higlights.png: 11 raw changes
Processed 10.23_patch_higlights.jpg: 0 raw changes
Processed 10.24_patch_higlights.jpg: 6 raw changes
Processed 10.25_patch_higlights.jpg: 24 raw changes
Processed 11.10_patch_higlights.jpg: 12 raw changes
Processed 11.11_patch_higlights.jpg: 17 raw changes
Processed 11.12_patch_higlights.png: 16 raw changes
Processed 11.13

,Champion,Buff/Nerf,Patch Number,Image File
0,Annie,Buff,10.10,10.10_patch_higlights.png
1,Irelia,Buff,10.10,10.10_patch_higlights.png
2,Nidalee,Buff,10.10,10.10_patch_higlights.png
3,Sivir,Buff,10.10,10.10_patch_higlights.png
4,Soraka,Buff,10.10,10.10_patch_higlights.png
